# PHA 完整流程示例

本 Notebook 展示个人健康 Agent（PHA）的完整工作流程。编排器协调三个专用 Agent。

1. **数据科学 Agent**分析数据及其趋势和规律。
2. **领域专家 Agent**提供医学解读和背景信息。
3. **健康教练 Agent**通过对话给出指导和建议。

## 工作流程

1. 用户提出一个健康问题。
2. 编排器判断由哪些 Agent 处理。
3. 用于辅助分析的 Agent 提供各自的结果。这一步可以并行执行。
4. 主 Agent 汇总结果并组织回复。
5. 用户收到个性化回答。

## 环境搭建

In [ ]:
import os
import sys

# 将工程根添加到路径
sys.path.insert(0, '..')

# --- Configuration ---
# 在此设定您的 API 密钥和提供者
API_KEY = "your-api-key-here"
# 提供方选项："基因","openai","人类"
PROVIDER = "gemini"

# 可选：Tavily API 键 领域专家 Agent (搜索).
TAVILY_API_KEY = "your-tavily-api-key-here"


In [ ]:
from pha.agents import (
    DataScienceAgent,
    DomainExpertAgent,
    HealthCoachAgent,
    MultiAgentOrchestrator,
    is_react_available,
    create_orchestrator,
)

print("All agents imported successfully!")
print(f"ReAct available for Domain Expert: {is_react_available()}")

## 加载健康数据样本

In [ ]:
import pandas as pd

# 装入样本数据
summary_df = pd.read_csv('../data/sample/summary.csv')
activities_df = pd.read_csv('../data/sample/activities.csv')
profile_df = pd.read_csv('../data/sample/profile.csv')
population_df = pd.read_csv('../data/sample/population_percentiles.csv')

print(f"Loaded data:")
print(f"  - Summary: {len(summary_df)} days")
print(f"  - Activities: {len(activities_df)} records")
print(f"  - Profile: {profile_df.to_dict('records')[0]}")

## 初始化各个 Agent

In [ ]:
# 初始化 数据科学 Agent (美国英语).
ds_agent = DataScienceAgent()
ds_agent.configure(api_key=API_KEY, provider=PROVIDER) # 隐含使用全局配置
ds_agent.load_dataframes({
    'summary': summary_df,
    'activities': activities_df,
    'profile': profile_df,
    'population': population_df,
})
print("✓ Data Science Agent initialized")

In [ ]:
# 初始化领域专家 Agent(如果ReAct可用).
de_agent = None
if is_react_available():
    import glob
    de_agent = DomainExpertAgent(
        search_backend='tavily',  # 或“ Duckduckgo” 免费搜索
        tavily_api_key=TAVILY_API_KEY,
    )
    exemplar_files = glob.glob('../few_shots/*.ipynb')
    de_agent.get_agent(
        api_key=API_KEY,
        provider=PROVIDER,
        exemplar_files=exemplar_files,
    )
    print(f"✓ Domain Expert Agent initialized with {len(exemplar_files)} exemplars")
else:
    print("⚠ Domain Expert Agent unavailable (install onetwo for ReAct support)")


In [ ]:
# 初始化 健康教练 Agent (美国英语).
hc_agent = HealthCoachAgent(simple_mode=True)
hc_agent.configure(api_key=API_KEY, provider=PROVIDER)
print("✓ Health Coach Agent initialized")

## 初始化编排器

In [ ]:
# 创建 编排器 并连接所有代理
orchestrator = MultiAgentOrchestrator(debug_verbose=True)
orchestrator.configure(api_key=API_KEY, provider=PROVIDER)
orchestrator.set_agents(
    data_science_agent=ds_agent,
    domain_expert_agent=de_agent,
    health_coach_agent=hc_agent,
)

print("\n✓ Orchestrator initialized!")
print(f"  Agents: {orchestrator.agent_name_list}")

## 测试完整流程

In [ ]:
# 问一个全面的健康问题
response = orchestrator.respond(
    "Based on my health data, how is my sleep quality and what can I do to improve it?"
)

print("\n" + "="*60)
print("RESPONSE:")
print("="*60)
print(response)

In [ ]:
# 请检查使用过的团队结构
print("Team Structure Used:")
print(orchestrator.get_team_structure())

In [ ]:
# 检查个体特工的见解
print("\nAgent Insights:")
insights = orchestrator.get_agent_insights()
for agent, insight in insights.items():
    if insight:
        print(f"\n--- {agent.upper()} ---")
        print(insight[:500] + "..." if len(insight) > 500 else insight)

## 多轮对话

In [ ]:
# 继续谈话
response = orchestrator.respond(
    "How does my activity level compare to others my age?"
)

print("\n" + "="*60)
print("RESPONSE:")
print("="*60)
print(response)

In [ ]:
# 询问趋势
response = orchestrator.respond(
    "Have my resting heart rate or steps been trending up or down lately?"
)

print("\n" + "="*60)
print("RESPONSE:")
print("="*60)
print(response)

## 查看对话历史

In [ ]:
print("Conversation History:")
print(orchestrator.get_conversation_history())

## 使用工厂函数创建编排器

可以通过工厂函数直接创建配置好的编排器。

In [ ]:
# 使用工厂功能快速设置
quick_orchestrator = create_orchestrator(
    data_science_agent=ds_agent,
    domain_expert_agent=de_agent,
    health_coach_agent=hc_agent,
    debug_verbose=False,  # 静音模式
)

response = quick_orchestrator.respond("What's my average daily step count?")
print(response)

## 概要

PHA 编排器负责以下工作。

1. 把问题自动交给最合适的 Agent。
2. 并行调用辅助 Agent，提高处理效率。
3. 通过主 Agent 综合各方分析。
4. 在多轮交互中维护对话上下文。
5. 汇总数据分析、领域解读和健康指导，形成完整回答。

### 架构

```mermaid
flowchart TD
  Q["用户提问"] --> O["编排器"]
  O --> D["数据科学 Agent"]
  O --> E["领域专家 Agent"]
  O --> C["健康教练 Agent"]
  D --> R["汇总回复"]
  E --> R
  C --> R
  R --> U["用户收到回答"]
```